In [ ]:
import pandas as pd
import json

# Load the dataset
file_path = 'COVID-19_Vaccinations_in_the_United_States,County_20251119.csv'
# Use low_memory=False to avoid mixed type warnings if the file is large
df = pd.read_csv(file_path, low_memory=False)

# Display first few rows to inspect
df.head()

In [ ]:
# Data Cleaning

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract Year
df['Year'] = df['Date'].dt.year

# Columns to clean (remove commas and convert to numeric)
cols_to_clean = ['Series_Complete_Pop_Pct', 'Series_Complete_Yes', 'Census2019', 'Completeness_pct']

for col in cols_to_clean:
    # Ensure column is string before replacing comma, then convert to numeric
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# 1. Aggregate Vaccination Rates per County by Year
# We take the maximum vaccination rate for each year to represent the status by the end of that year.
# We also need to keep FIPS to link with the map.

# Ensure FIPS is string and padded
df['FIPS'] = df['FIPS'].astype(str).str.zfill(5)

# Group by FIPS and Year
county_vax_by_year = df.groupby(['FIPS', 'Recip_State', 'Recip_County', 'Year'])[['Series_Complete_Pop_Pct', 'Completeness_pct']].max().reset_index()

print("County Level Aggregation (First 5 rows):")
display(county_vax_by_year.head())

In [ ]:
# 2. Aggregate Vaccination Rates per State by Year
# Calculate weighted average for states
county_yearly_max = df.groupby(['FIPS', 'Recip_State', 'Year'])[['Series_Complete_Yes', 'Census2019']].max().reset_index()
state_agg = county_yearly_max.groupby(['Recip_State', 'Year'])[['Series_Complete_Yes', 'Census2019']].sum().reset_index()
state_agg['Series_Complete_Pop_Pct'] = (state_agg['Series_Complete_Yes'] / state_agg['Census2019']) * 100

# Create a mapping for State Abbreviation to FIPS (Numeric) to match TopoJSON IDs
# This mapping covers the 50 states + DC + PR
state_abbr_to_fips = {
    'AL': 1, 'AK': 2, 'AZ': 4, 'AR': 5, 'CA': 6, 'CO': 8, 'CT': 9, 'DE': 10, 'DC': 11,
    'FL': 12, 'GA': 13, 'HI': 15, 'ID': 16, 'IL': 17, 'IN': 18, 'IA': 19, 'KS': 20,
    'KY': 21, 'LA': 22, 'ME': 23, 'MD': 24, 'MA': 25, 'MI': 26, 'MN': 27, 'MS': 28,
    'MO': 29, 'MT': 30, 'NE': 31, 'NV': 32, 'NH': 33, 'NJ': 34, 'NM': 35, 'NY': 36,
    'NC': 37, 'ND': 38, 'OH': 39, 'OK': 40, 'OR': 41, 'PA': 42, 'RI': 44, 'SC': 45,
    'SD': 46, 'TN': 47, 'TX': 48, 'UT': 49, 'VT': 50, 'VA': 51, 'WA': 53, 'WV': 54,
    'WI': 55, 'WY': 56, 'PR': 72
}

state_agg['FIPS'] = state_agg['Recip_State'].map(state_abbr_to_fips)

# Filter out states that didn't match (e.g., territories not in our map)
state_agg = state_agg.dropna(subset=['FIPS'])
state_agg['FIPS'] = state_agg['FIPS'].astype(int).astype(str) # Ensure it matches string IDs if needed, or int. TopoJSON often uses IDs.

print("State Level Aggregation (First 5 rows):")
display(state_agg.head())

In [ ]:
# 3. Export to JSON Structure for Web App
output_data = {
    "years": sorted(df['Year'].unique().tolist()),
    "counties": {},
    "states": {}
}

# Populate Counties
for _, row in county_vax_by_year.iterrows():
    fips = row['FIPS']
    year = str(row['Year'])
    if fips not in output_data["counties"]:
        output_data["counties"][fips] = {}
    
    output_data["counties"][fips][year] = {
        "rate": row['Series_Complete_Pop_Pct'],
        "completeness": row['Completeness_pct'],
        "name": row['Recip_County'],
        "state": row['Recip_State']
    }

# Populate States
for _, row in state_agg.iterrows():
    fips = str(row['FIPS'])
    year = str(row['Year'])
    if fips not in output_data["states"]:
        output_data["states"][fips] = {}
        
    output_data["states"][fips][year] = {
        "rate": row['Series_Complete_Pop_Pct'],
        "abbr": row['Recip_State']
    }

# Save to JSON file
with open('vaccination_data.json', 'w') as f:
    json.dump(output_data, f)

print("Data exported to vaccination_data.json")